--- 
# **[실습]**

- 지금까지 학습한 여러 기법들을 선택하여, RAG 답변을 생성하는 체인을 구성합니다. 

`(1) 기본 검색기 설정`

- Semantic Search, Keyword Search, Hybrid Search 검색기를 직접 정의합니다. 
- 쿼리 확장 도구 적용을 검토합니다.

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 벡터 저장소 로드
chroma_db = Chroma(
    collection_name="db_korean_cosine_metadata",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

# Semantic Search 검색기 생성
semantic_retriever = chroma_db.as_retriever(
    search_kwargs={"k": 5}
)

### Keyword Search 검색기 생성

In [ ]:
from langchain.retrievers import BM25Retriever
from kiwipiepy import Kiwi
from langchain_core.documents import Document

# 한국어 토크나이저 초기화
kiwi = Kiwi()

def bm25_process_func(text, kiwi_model=kiwi):
    """BM25용 한국어 토큰화 함수"""
    return [t.form for t in kiwi_model.tokenize(text)]

chroma_db.get().keys()
documents = chroma_db.get()["documents"]
metadatas = chroma_db.get()["metadatas"]
docs = [Document(page_content=content, metadata=meta) for content, meta in zip(documents, metadatas)]

# BM25 검색기 생성
bm25_retriever = BM25Retriever.from_documents(
    documents=docs,
    preprocess_func=lambda x: bm25_process_func(x, kiwi_model=kiwi),
    k=5
)

### 하이브리드 검색기 생성 (Semantic + Keyword)

In [ ]:
from langchain.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[semantic_retriever, bm25_retriever], 
    weights=[0.5, 0.5]  # 각 검색기 가중치
)

### 쿼리 확장도구 적용

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 멀티쿼리 프롬프트 템플릿
multiquery_template = """
다음 질문에 대해 5개의 다른 버전으로 다시 작성해주세요:
{question}

각 줄에 하나씩 작성하세요:
"""

# 멀티쿼리 체인 생성
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
multiquery_chain = (
    ChatPromptTemplate.from_template(multiquery_template) 
    | llm 
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

# Multi-Query 검색기 생성
multi_query_retriever = MultiQueryRetriever(
    retriever=semantic_retriever,
    llm_chain=multiquery_chain,
    parser_key="lines"
)

# HyDE 프롬프트 템플릿
hyde_template = """
다음 질문에 대한 가상의 문서를 작성해주세요:
질문: {question}

가상 문서:
"""

# HyDE 체인 생성
hyde_chain = (
    ChatPromptTemplate.from_template(hyde_template)
    | llm
    | StrOutputParser()
)

# HyDE + 검색 파이프라인
def hyde_search(query):
    # 1. 가상 문서 생성
    hypothetical_doc = hyde_chain.invoke({"question": query})
    
    # 2. 가상 문서로 검색
    retrieved_docs = semantic_retriever.invoke(hypothetical_doc)
    
    return retrieved_docs

# 쿼리 리포뮬레이션 템플릿
reformulation_template = """
다음 질문을 검색 성능을 향상시키기 위해 다시 작성해주세요:
[질문]
{question}

다음 방식으로 질문을 재작성하세요:
1. 동의어 추가
2. 더 구체적인 키워드 포함
3. 관련된 개념 확장

[재작성된 질문]
"""

# 리포뮬레이션 체인
reformulation_chain = (
    ChatPromptTemplate.from_template(reformulation_template)
    | llm
    | StrOutputParser()
)

# 리포뮬레이션 검색기
reformulation_retriever = reformulation_chain | semantic_retriever

`(2) 검색기법 고도화`

- Rerank, Comporession 기법을 적용합니다. 
- Pipeline Compressor로 연결하여 구성합니다. 

### Rerank

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import LLMListwiseRerank

# Cross-Encoder 모델 초기화
cross_encoder_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")

# Cross-Encoder Reranker 생성
cross_encoder_reranker = CrossEncoderReranker(
    model=cross_encoder_model, 
    top_n=3
)

# Cross-Encoder 기반 검색기
cross_encoder_retriever = ContextualCompressionRetriever(
    base_compressor=cross_encoder_reranker,
    base_retriever=hybrid_retriever,
)

# LLM 기반 Reranker
llm_reranker = LLMListwiseRerank.from_llm(llm, top_n=3)

# LLM Reranker 검색기
llm_reranker_retriever = ContextualCompressionRetriever(
    base_compressor=llm_reranker,
    base_retriever=hybrid_retriever,
)

### Compression

In [ ]:
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain.retrievers.document_compressors import LLMChainFilter
from langchain.retrievers.document_compressors import LLMChainExtractor

# LLM 필터 생성
context_filter = LLMChainFilter.from_llm(llm)

# LLM 필터 검색기
llm_filter_retriever = ContextualCompressionRetriever(
    base_compressor=context_filter,
    base_retriever=cross_encoder_retriever,
)


# LLM 추출기 생성
llm_extractor = LLMChainExtractor.from_llm(llm)

# LLM 추출기 검색기
llm_extractor_retriever = ContextualCompressionRetriever(
    base_compressor=llm_extractor,
    base_retriever=cross_encoder_retriever,
)


# 중복 제거 필터
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)

# 유사도 필터
relevant_filter = EmbeddingsFilter(
    embeddings=embeddings, 
    similarity_threshold=0.4
)

# Re-ranking 모델
re_ranker = LLMListwiseRerank.from_llm(llm, top_n=2)

# Pipeline Compressor 생성
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[redundant_filter, relevant_filter, re_ranker]
)

# Pipeline 기반 검색기
pipeline_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=hybrid_retriever,
)

`(3) RAG 체인 연결`

- 검색기, 프롬프트, LLM을 LCEL로 연결하여 RAG Chain을 구성합니다. 
- 다양한 쿼리를 입력하고, 생성된 답변의 품질을 평가합니다. 

### RAG 체인 연결

기본체인

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# RAG 프롬프트 템플릿
rag_template = """
다음 컨텍스트를 기반으로 질문에 답변해주세요:

[컨텍스트]
{context}

[질문]
{question}

[답변]
"""

# 문서 포맷팅 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG 체인 구성 (LCEL)
rag_chain = (
    {
        "context": RunnableLambda(lambda x: x["question"]) | pipeline_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | ChatPromptTemplate.from_template(rag_template)
    | llm
    | StrOutputParser()
)

HyDE + Pipeline + RAG

LLM 호출 증가로 처리 시간이나 비용은 늘어나지만
HyDE를 이용해 가상 문서를 생성 후 사용하여 성능 향상

In [ ]:
def create_hyde_rag_chain(hyde_chain, retriever, llm):
    def hyde_retrieve_and_format(input_dict):
        question = input_dict["question"]
        
        # 1. HyDE: 가상 문서 생성
        hypothetical_doc = hyde_chain.invoke({"question": question})
        
        # 2. Pipeline 검색
        docs = retriever.invoke(hypothetical_doc)
        
        # 3. 컨텍스트 포맷팅
        context = format_docs(docs)
        
        return {"context": context, "question": question}
    
    hyde_rag_chain = (
        RunnableLambda(hyde_retrieve_and_format)
        | ChatPromptTemplate.from_template(rag_template)
        | llm
        | StrOutputParser()
    )
    return hyde_rag_chain

# 완전한 HyDE-RAG 체인 생성
hyde_rag_chain = create_hyde_rag_chain(hyde_chain, pipeline_retriever, llm)

### 쿼리 테스트 및 평가

In [ ]:
# 다양한 유형의 테스트 쿼리
test_queries = [
    "리비안은 언제 사업을 시작했나요?",
    "테슬라의 경영진을 분석해주세요.",
    "리비안의 사업 경쟁력은 어디서 나오나요?",
    "테슬라 트럭 모델이 있나요?",
    "전기차 시장의 주요 경쟁사는 누구인가요?"
]

# 각 쿼리별 답변 생성 및 비교
for query in test_queries:
    print(f"\n질문: {query}")
    print("-" * 50)
    
    try:
        # 기본 RAG
        basic_answer = rag_chain.invoke({"question": query})
        print(f"기본 RAG: {basic_answer}")
        
        # 고급 RAG
        advanced_answer = hyde_rag_chain.invoke({"question": query})
        print(f"고급 RAG: {advanced_answer}")
        
    except Exception as e:
        print(f"오류 발생: {e}")
        
        # 대안: 직접 호출 방식
        try:
            # 직접 검색 및 답변 생성
            docs = pipeline_retriever.invoke(query)
            context = format_docs(docs)
            
            direct_answer = llm.invoke([
                {"role": "system", "content": "주어진 컨텍스트를 바탕으로 질문에 정확하게 답변하세요."},
                {"role": "user", "content": f"컨텍스트:\n{context}\n\n질문: {query}"}
            ]).content
            
            print(f"직접 RAG: {direct_answer}")
            
        except Exception as e2:
            print(f"대안 방법도 실패: {e2}")
    
    print("=" * 100)

In [ ]:
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from krag.evaluators import RougeOfflineRetrievalEvaluators

def evaluate_qa_test(df_qa_test: pd.DataFrame, retriever: BaseRetriever, k=2) -> dict:
    """
    테스트 데이터셋에 대한 검색 결과 평가
    """

    context_docs = []
    retrieved_docs = []

    df_test = df_qa_test.copy()
    
    print(f"📊 {len(df_test)}개 테스트 케이스 평가 시작...")
    
    for idx, row in df_test.iterrows():
        try:
            # 1. 질문과 정답 컨텍스트 추출
            question = row['user_input']  # 또는 row['question']
            
            # reference_contexts가 문자열인 경우 파싱
            if isinstance(row['reference_contexts'], str):
                context_list = eval(row['reference_contexts'])
            else:
                context_list = row['reference_contexts']
            
            # 2. 정답 문서들을 Document 객체로 변환
            context_doc = [Document(page_content=doc) for doc in context_list]
            context_docs.append(context_doc)
            
            # 3. 검색기로 문서 검색
            retrieved_doc = retriever.invoke(question)  
            retrieved_docs.append(retrieved_doc)
            
            print(f"✅ {idx+1}/{len(df_test)}: {question[:50]}...")
            
        except Exception as e:
            print(f"❌ {idx+1}/{len(df_test)} 오류: {e}")
            # 빈 리스트 추가하여 인덱스 맞춤
            context_docs.append([])
            retrieved_docs.append([])

    # 4. 평가자 인스턴스 생성
    evaluator = RougeOfflineRetrievalEvaluators(
        actual_docs=context_docs,      # 정답 문서들
        predicted_docs=retrieved_docs, # 검색된 문서들
        match_method='rouge1',         # ROUGE-1 기반 매칭
        threshold=0.8,                 # 매칭 임계값 (0.8 이상이면 일치로 판단)
    )

    # 5. 평가지표 계산
    hit_rate = evaluator.calculate_hit_rate(k=k)['hit_rate']
    mrr = evaluator.calculate_mrr(k=k)['mrr']
    map_score = evaluator.calculate_map(k=k)['map']
    ndcg = evaluator.calculate_ndcg(k=k)['ndcg']

    print(f"\n📈 K={k} 평가 결과")
    print("-"*50)
    print(f"Hit Rate: {hit_rate:.3f}")
    print(f"MRR: {mrr:.3f}")
    print(f"MAP: {map_score:.3f}")
    print(f"NDCG: {ndcg:.3f}")
    print("="*50)

    result = {
        'hit_rate': hit_rate,
        'mrr': mrr,
        'map': map_score,
        'ndcg': ndcg,
    }

    return pd.Series(result)

### 사용

In [ ]:
class AdvancedRAGSystem:
    def __init__(self):
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        self.llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
        self.setup_retrievers()
        self.setup_chains()
    
    def setup_retrievers(self):
        # 기본 검색기들
        self.semantic_retriever = semantic_retriever
        self.bm25_retriever = bm25_retriever
        
        # 하이브리드 검색기
        self.hybrid_retriever = EnsembleRetriever(
            retrievers=[self.semantic_retriever, self.bm25_retriever],
            weights=[0.6, 0.4]
        )
        
        # Pipeline Compressor
        redundant_filter = EmbeddingsRedundantFilter(embeddings=self.embeddings)
        relevant_filter = EmbeddingsFilter(embeddings=self.embeddings, similarity_threshold=0.4)
        re_ranker = LLMListwiseRerank.from_llm(self.llm, top_n=3)
        
        pipeline_compressor = DocumentCompressorPipeline(
            transformers=[redundant_filter, relevant_filter, re_ranker]
        )
        
        self.final_retriever = ContextualCompressionRetriever(
            base_compressor=pipeline_compressor,
            base_retriever=self.hybrid_retriever,
        )
    
    def setup_chains(self):
        # HyDE 체인
        hyde_template = """질문: {question}에 대한 가상의 답변 문서를 작성해주세요."""
        self.hyde_chain = (
            ChatPromptTemplate.from_template(hyde_template)
            | self.llm
            | StrOutputParser()
        )
        
        # RAG 체인 - 안전한 버전
        rag_template = """
        컨텍스트: {context}
        질문: {question}
        
        위 컨텍스트를 바탕으로 질문에 정확하고 자세하게 답변해주세요.
        """
        
        def safe_retrieve_and_format(question):
            try:
                docs = self.final_retriever.invoke(question)
                return self.format_docs(docs)
            except Exception as e:
                print(f"검색 오류: {e}")
                return "검색 결과를 가져올 수 없습니다."
        
        self.rag_chain = (
            RunnablePassthrough.assign(
                context=RunnableLambda(lambda x: safe_retrieve_and_format(x["question"]))
            )
            | ChatPromptTemplate.from_template(rag_template)
            | self.llm
            | StrOutputParser()
        )
    
    def format_docs(self, docs):
        return "\n\n".join(f"[출처: {doc.metadata.get('source', 'Unknown')}]\n{doc.page_content}" for doc in docs)
    
    def query(self, question, use_hyde=True):
        try:
            if use_hyde:
                # HyDE 사용
                hypothetical_doc = self.hyde_chain.invoke({"question": question})
                docs = self.final_retriever.invoke(hypothetical_doc)
                context = self.format_docs(docs)
                
                messages = [
                    {"role": "system", "content": "주어진 컨텍스트를 바탕으로 정확하게 답변하세요."},
                    {"role": "user", "content": f"컨텍스트: {context}\n\n질문: {question}"}
                ]
                return self.llm.invoke(messages).content
            else:
                # 일반 RAG
                return self.rag_chain.invoke({"question": question})
        except Exception as e:
            return f"답변 생성 중 오류가 발생했습니다: {e}"
    
    def simple_query(self, question):
        """간단한 RAG 실행 (오류 처리 포함)"""
        try:
            docs = self.final_retriever.invoke(question)
            context = self.format_docs(docs)
            
            prompt = f"""
            다음 컨텍스트를 바탕으로 질문에 답변해주세요:
            
            컨텍스트:
            {context}
            
            질문: {question}
            
            답변:
            """
            
            return self.llm.invoke(prompt).content
        except Exception as e:
            return f"오류 발생: {e}"

# 사용 예시
rag_system = AdvancedRAGSystem()

# 테스트
def test_rag_system():
    test_queries = [
        "리비안은 언제 사업을 시작했나요?",
        "테슬라의 경영진을 분석해주세요.",
        "리비안의 사업 경쟁력은 어디서 나오나요?",
        "테슬라 트럭 모델이 있나요?",
        "전기차 시장의 주요 경쟁사는 누구인가요?"
    ]
    
    for query in test_queries:
        print(f"\n질문: {query}")
        print("-" * 50)
        
        # 간단한 방법으로 테스트
        answer = rag_system.simple_query(query)
        print(f"답변: {answer}")
        
        print("=" * 100)

# 테스트 실행
test_rag_system()